# Manuscript figures 2–4

| Figure | Panels | Reads |
|---|---|---|
| 2 — *Klebsiella* read fraction | a genus fraction, b top non-*Klebsiella* | `quality_control_and_routing.tsv` + `../pipeline_tables/kraken2_summary.tsv` |
| 3 — assembly strategy comparison | a contigs, b total length | `quality_control_and_routing.tsv` + `../pipeline_tables/assembly_stats.tsv` |
| 4 — mapping rate distribution | single | `quality_control_and_routing.tsv` |

In [2]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, FixedFormatter, NullLocator

TABLES = Path("../pipeline_tables")   # the stage summary tables this repo ships

SRC = {
    "s5": TABLES / "quality_control_and_routing.tsv",
    "asm_stats": TABLES / "assembly_stats.tsv",
    "kraken2": TABLES / "kraken2_summary.tsv",
}
missing = [f"{k} -> {v}" for k, v in SRC.items() if not Path(v).exists()]
if missing:
    raise SystemExit("missing inputs:\n  " + "\n  ".join(missing))

# the routing table every figure reads
s5 = pd.read_csv(SRC["s5"], sep="\t")

FIGDIR = Path("figures")
DATADIR = FIGDIR / "figure_data"
FIGDIR.mkdir(parents=True, exist_ok=True)
DATADIR.mkdir(parents=True, exist_ok=True)

SEED = 20260810          # jitter RNG seed; changing it changes the figures
print(f"{len(SRC)} inputs present | jitter seed {SEED}")

3 inputs present | jitter seed 20260810


## Shared styling

In [3]:
MM = 1 / 25.4                       # Scientific Data: 89 mm single column, 183 mm double

COLOUR = {
    "ont": "#0072B2",               # Nanopore-only
    "illumina": "#D55E00",          # Illumina-only
    "hybrid": "#009E73",            # hybrid
    "released": "#4D4D4D",          # released libraries / kraken2 PASS
    "excluded": "#D55E00",          # excluded libraries / kraken2 FAIL
    "threshold": "#999999",         # threshold lines
}


def apply_style():
    '''Scientific Data house style. Called by every figure.'''
    plt.rcdefaults()
    plt.rcParams.update({
        # Arial is not installed here; Liberation Sans is metric-compatible with it and
        # both output formats keep the text editable, so the typesetter can substitute.
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
        "font.size": 8,
        "axes.labelsize": 8,
        "axes.titlesize": 8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 7,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.linewidth": 0.8,
        "axes.grid": False,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 2,
        "ytick.major.size": 2,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "lines.linewidth": 0.8,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "pdf.fonttype": 42,          # embed as TrueType, text stays selectable
        "ps.fonttype": 42,
        "svg.fonttype": "none",      # SVG keeps live <text>, not outlines
    })


def panel_letter(ax, letter, x=-0.16, y=1.06):
    '''9 pt bold lowercase letter, top left, outside the axes.'''
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=9, fontweight="bold",
            ha="left", va="bottom")


def threshold_line(ax, x):
    '''Dashed grey rule behind the data. Uncaptioned: the figure legend names it.'''
    ax.axvline(x, color=COLOUR["threshold"], linestyle="--", linewidth=0.8, zorder=0)


def save_figure(fig, stem):
    '''PDF + SVG + 300 dpi PNG preview, all with the same bounding box.'''
    written = []
    for ext in ("pdf", "svg", "png"):
        path = FIGDIR / f"{stem}.{ext}"
        fig.savefig(path, dpi=300, bbox_inches="tight", pad_inches=0.02)
        written.append(f"{path}  ({path.stat().st_size / 1024:.0f} kB)")
    plt.close(fig)
    print("wrote:\n  " + "\n  ".join(written))


def dump_panel(frame, name):
    '''The exact plotted values, one CSV per panel, for checking and for Zenodo.'''
    path = DATADIR / f"{name}.csv"
    frame.to_csv(path, index=False)
    print(f"panel data: {path}  ({len(frame)} rows, {len(frame.columns)} cols)")


apply_style()
_face = font_manager.findfont(font_manager.FontProperties(
    family=plt.rcParams["font.sans-serif"]))
print(f"style applied | sans-serif resolves to {Path(_face).stem}")

style applied | sans-serif resolves to LiberationSans-Regular


## Figure 2 — *Klebsiella* read fraction and contamination flags

In [4]:
kraken = pd.read_csv(SRC["kraken2"], sep="\t")
assert len(kraken) == 316, f"kraken2 summary has {len(kraken)} rows, expected 316"

fig2 = s5[["library", "isolate", "klebsiella_genus_read_fraction", "top_non_klebsiella_genus",
           "top_non_klebsiella_genus_fraction", "kraken2_status", "released",
           "internal_qc_decision"]].copy()
fig2 = fig2.merge(kraken[["sample", "status", "flags"]].rename(columns={"sample": "library"}),
                  on="library", how="left", validate="1:1")
assert fig2.status.notna().all(), "kraken2 summary does not cover every library"
assert (fig2.status == fig2.kraken2_status).all(), "kraken2 status disagrees with the routing table"

# Are the fractions proportions or percentages? Assert rather than assume.
for col in ["klebsiella_genus_read_fraction", "top_non_klebsiella_genus_fraction"]:
    assert fig2[col].notna().all(), f"null {col}"
    assert fig2[col].max() <= 1.0, f"{col} max is {fig2[col].max()}, not a proportion"
    assert fig2[col].min() > 0, f"{col} has a non-positive value, no log complement"
LOW_YIELD, NON_KLEB = 0.50, 0.05

flags = fig2["flags"].fillna("")
fig2["low_yield"] = flags.str.contains("LOW_KLEBSIELLA")
fig2["non_klebsiella"] = flags.str.contains("NON_KLEBSIELLA_CONTAMINATION")
fig2["mixed_klebsiella"] = flags.str.contains("MIXED_KLEBSIELLA")
# the triangles: every library adjudicated as a two-species Klebsiella mixture, which adds
# KSB2_5D — its second species is 9.9% of reads, under the 20% flag, but its assembly ran
# to 10.4 Mb and it went to decontamination with the flagged ones
fig2["mixed_marker"] = (fig2.mixed_klebsiella
                        | (fig2.internal_qc_decision == "REDO_DECONTAM_KLEBSIELLA"))

In [5]:
apply_style()
rng = np.random.default_rng(SEED)

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(183 * MM, 65 * MM))

fig2["top_non_kleb_pct"] = 100 * fig2.top_non_klebsiella_genus_fraction
fig2["y_jitter_a"] = rng.uniform(0.05, 0.95, size=len(fig2))
fig2["y_jitter_b"] = rng.uniform(0.05, 0.95, size=len(fig2))

STATUS = [("PASS", COLOUR["released"]), ("FAIL", COLOUR["excluded"])]

# --- panel a: Klebsiella genus read fraction, logit ---------------------------------
# Logit rather than the log of the complement, for the same reason as Fig. 4 and more
# sharply: six libraries sit between 0.7% and 29.5%, and a log-complement axis stacks
# them into one blob against the left limit — exactly the libraries the panel is about.
for status, colour in STATUS:
    sub = fig2[(fig2.kraken2_status == status) & ~fig2.mixed_marker]
    ax_a.scatter(sub.klebsiella_genus_read_fraction, sub.y_jitter_a, s=5, alpha=0.6,
                 linewidths=0, color=colour, zorder=2)
mixed = fig2[fig2.mixed_marker]
ax_a.scatter(mixed.klebsiella_genus_read_fraction, mixed.y_jitter_a, s=13, marker="^",
             facecolors="none", edgecolors=COLOUR["excluded"], linewidths=0.7,
             alpha=0.9, zorder=3)

ax_a.set_xscale("logit")
FRAC_TICKS = [1, 10, 50, 90, 99]
ax_a.xaxis.set_major_locator(FixedLocator([t / 100 for t in FRAC_TICKS]))
ax_a.xaxis.set_major_formatter(FixedFormatter([f"{t:g}" for t in FRAC_TICKS]))
ax_a.xaxis.set_minor_locator(NullLocator())
ax_a.set_xlim(0.004, 0.996)           # clear of the 0.66% minimum and the 99.31% maximum
ax_a.set_xlabel("$\\it{Klebsiella}$ genus read fraction (%)")
threshold_line(ax_a, LOW_YIELD)

# --- panel b: highest non-Klebsiella genus fraction, direct log ---------------------
for status, colour in STATUS:
    sub = fig2[fig2.kraken2_status == status]
    ax_b.scatter(sub.top_non_kleb_pct, sub.y_jitter_b, s=5, alpha=0.6, linewidths=0,
                 color=colour, zorder=2)

ax_b.set_xscale("log")
B_TICKS = [0.1, 1, 5, 10, 50, 100]
ax_b.xaxis.set_major_locator(FixedLocator(B_TICKS))
ax_b.xaxis.set_major_formatter(FixedFormatter([f"{t:g}" for t in B_TICKS]))
ax_b.xaxis.set_minor_locator(NullLocator())
ax_b.set_xlim(0.04, 190)
ax_b.set_xlabel("Highest non-$\\it{Klebsiella}$ genus fraction (%)")
threshold_line(ax_b, 100 * NON_KLEB)

for ax, letter in [(ax_a, "a"), (ax_b, "b")]:
    ax.set_ylim(-0.05, 1.05)
    ax.yaxis.set_visible(False)
    ax.spines["left"].set_visible(False)
    panel_letter(ax, letter, x=-0.06)

# one legend for both panels, a single row under the axes
handles = [
    Line2D([], [], marker="o", linestyle="", markersize=3.2, alpha=0.6,
           color=COLOUR["released"], label="PASS"),
    Line2D([], [], marker="o", linestyle="", markersize=3.2, alpha=0.6,
           color=COLOUR["excluded"], label="flagged"),
    Line2D([], [], marker="^", linestyle="", markersize=4.0, markerfacecolor="none",
           markeredgecolor=COLOUR["excluded"], markeredgewidth=0.7,
           label="mixed $\\it{Klebsiella}$"),
]
fig.subplots_adjust(wspace=0.16, bottom=0.22)
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=3,
           frameon=False, fontsize=7, handletextpad=0.3, columnspacing=1.5,
           borderaxespad=0.0)
save_figure(fig, "Fig2_klebsiella_read_fraction")
dump_panel(fig2[["library", "isolate", "klebsiella_genus_read_fraction", "kraken2_status",
                 "low_yield", "mixed_klebsiella", "mixed_marker", "released",
                 "y_jitter_a"]],
           "Fig2_panel_a_klebsiella_fraction")
dump_panel(fig2[["library", "isolate", "top_non_klebsiella_genus",
                 "top_non_klebsiella_genus_fraction", "top_non_kleb_pct", "non_klebsiella",
                 "kraken2_status", "released", "y_jitter_b"]],
           "Fig2_panel_b_top_non_klebsiella")

wrote:
  figures/Fig2_klebsiella_read_fraction.pdf  (33 kB)
  figures/Fig2_klebsiella_read_fraction.svg  (90 kB)
  figures/Fig2_klebsiella_read_fraction.png  (112 kB)
panel data: figures/figure_data/Fig2_panel_a_klebsiella_fraction.csv  (316 rows, 9 cols)
panel data: figures/figure_data/Fig2_panel_b_top_non_klebsiella.csv  (316 rows, 9 cols)


## Figure 3 — assembly strategy comparison

In [6]:
asm = pd.read_csv(SRC["asm_stats"], sep="\t")
asm["library"] = asm.file.str.split("/").str[1]
asm["strategy"] = asm.file.str.split("/").str[2]
illumina = (asm[asm.strategy == "illumina_unicycler"]
            .set_index("library")[["num_seqs", "sum_len"]]
            .rename(columns={"num_seqs": "illumina_contigs", "sum_len": "illumina_total_len"}))

released = s5[s5.released == "yes"].copy()
fig3 = released.merge(illumina, left_on="library", right_index=True,
                      how="left", validate="1:1")

# the join is the one step outside the supplementary tables, so check it explicitly
assert len(fig3) == 294, f"join yielded {len(fig3)} rows, expected 294"
null_cols = {c: int(fig3[c].isna().sum())
             for c in ["ont_contigs", "ont_total_len", "hybrid_contigs", "hybrid_total_len",
                       "illumina_contigs", "illumina_total_len"] if fig3[c].isna().any()}
assert not null_cols, f"nulls after join: {null_cols}"

STRATEGIES = [("Nanopore-only", "ont", "ont_contigs", "ont_total_len"),
              ("Illumina-only", "illumina", "illumina_contigs", "illumina_total_len"),
              ("hybrid", "hybrid", "hybrid_contigs", "hybrid_total_len")]

EXPECT_F3 = {                        # label: (contig mean, median, max, length mean Mbp)
    "Nanopore-only": (2.8, 2, 13, 5.53),
    "Illumina-only": (112.5, 95, 390, 5.45),
    "hybrid":        (2.8, 2, 16, 5.53),
}

In [7]:
apply_style()
rng = np.random.default_rng(SEED)

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(183 * MM, 75 * MM))
positions = np.arange(len(STRATEGIES))
JITTER = 0.28

panels = [(ax_a, "a", "ont_contigs", "illumina_contigs", "hybrid_contigs",
           "Contigs per assembly (log scale)"),
          (ax_b, "b", "ont_total_len", "illumina_total_len", "hybrid_total_len",
           "Total assembly length (Mbp)")]

dumps = {"a": [], "b": []}

for ax, letter, *_ , ylabel in panels:
    scale = 1.0 if letter == "a" else 1e-6
    for pos, (label, key, ccol, lcol) in zip(positions, STRATEGIES):
        col = ccol if letter == "a" else lcol
        values = fig3[col].astype(float).to_numpy() * scale
        offsets = rng.uniform(-JITTER, JITTER, size=values.size)

        ax.scatter(pos + offsets, values, s=4, alpha=0.35, linewidths=0,
                   color=COLOUR[key], zorder=2)
        ax.boxplot([values], positions=[pos], widths=0.32, showfliers=False,
                   patch_artist=True, zorder=3,
                   boxprops=dict(facecolor="white", edgecolor="black", linewidth=0.8),
                   medianprops=dict(color="black", linewidth=1.1),
                   whiskerprops=dict(color="black", linewidth=0.8),
                   capprops=dict(color="black", linewidth=0.8))

        dumps[letter].append(pd.DataFrame({
            "library": fig3.library.to_numpy(),
            "isolate": fig3.isolate.to_numpy(),
            "strategy": label,
            ("contigs" if letter == "a" else "total_length_mbp"): values,
            "x_position": pos,
            "x_jitter": offsets,
        }))

    ax.set_xticks(positions)
    ax.set_xticklabels([s[0] for s in STRATEGIES])
    ax.set_xlim(-0.6, len(STRATEGIES) - 0.4)
    ax.set_ylabel(ylabel)
    panel_letter(ax, letter, x=-0.18)

# panel a — log y, the range is 1 to 390
ax_a.set_yscale("log")
YTICKS = [1, 2, 5, 10, 20, 50, 100, 200, 400]
ax_a.yaxis.set_major_locator(FixedLocator(YTICKS))
ax_a.yaxis.set_major_formatter(FixedFormatter([str(t) for t in YTICKS]))
ax_a.yaxis.set_minor_locator(NullLocator())
ax_a.set_ylim(0.8, 560)

# panel b — linear, limits from the data. Starting at zero would flatten all three
# distributions into one line and hide the point that the strategies agree.
lengths = np.concatenate([fig3[c].to_numpy() / 1e6
                          for _, _, _, c in STRATEGIES])
ax_b.set_ylim(lengths.min() - 0.12, lengths.max() + 0.12)

fig.subplots_adjust(wspace=0.28)
save_figure(fig, "Fig3_assembly_strategy_comparison")
dump_panel(pd.concat(dumps["a"], ignore_index=True), "Fig3_panel_a_contigs")
dump_panel(pd.concat(dumps["b"], ignore_index=True), "Fig3_panel_b_total_length")

wrote:
  figures/Fig3_assembly_strategy_comparison.pdf  (43 kB)
  figures/Fig3_assembly_strategy_comparison.svg  (212 kB)
  figures/Fig3_assembly_strategy_comparison.png  (167 kB)
panel data: figures/figure_data/Fig3_panel_a_contigs.csv  (882 rows, 6 cols)
panel data: figures/figure_data/Fig3_panel_b_total_length.csv  (882 rows, 6 cols)


## Figure 4 — mapping rate distribution

In [9]:
fig4 = s5[["library", "isolate", "mapped_pct", "released", "kraken2_status",
           "internal_qc_decision"]].copy()
assert fig4.mapped_pct.notna().all(), "null mapped_pct"
assert fig4.mapped_pct.max() < 100, "a mapping rate of exactly 100% has no log complement"

rel3 = fig4[fig4.released == "yes"].mapped_pct
below90 = fig4[fig4.mapped_pct < 90]

In [10]:
apply_style()
rng = np.random.default_rng(SEED)

fig, ax = plt.subplots(figsize=(89 * MM, 65 * MM))

fig4["mapped_fraction"] = fig4.mapped_pct / 100      # the plotted x, logit needs (0, 1)
fig4["outcome"] = np.where(fig4.released == "yes", "released", "excluded")
fig4["y_jitter"] = rng.uniform(0.05, 0.95, size=len(fig4))

for outcome, colour in [("released", COLOUR["released"]), ("excluded", COLOUR["excluded"])]:
    sub = fig4[fig4.outcome == outcome]
    ax.scatter(sub.mapped_fraction, sub.y_jitter, s=5, alpha=0.6, linewidths=0,
               color=colour, zorder=2, label=outcome)

ax.set_xscale("logit")
# 95% is dropped from the spec's tick list — its label collides with 90% on a single
# column
RATE_TICKS = [10, 50, 90, 99, 99.9]
ax.xaxis.set_major_locator(FixedLocator([r / 100 for r in RATE_TICKS]))
ax.xaxis.set_major_formatter(FixedFormatter([f"{r:g}" for r in RATE_TICKS]))
ax.xaxis.set_minor_locator(NullLocator())
ax.set_xlim(0.05, 0.99995)            # clear of the 11.69% minimum and the 99.99% maximum
ax.set_xlabel("Illumina-to-Nanopore mapping rate (%)")

threshold_line(ax, 0.95)
threshold_line(ax, 0.90)

ax.set_ylim(-0.05, 1.18)
ax.yaxis.set_visible(False)
ax.spines["left"].set_visible(False)

ax.legend(loc="upper left", frameon=False, fontsize=7, handletextpad=0.3,
          borderaxespad=0.0, labelspacing=0.3, markerscale=1.8)

save_figure(fig, "Fig4_mapping_rate_distribution")
dump_panel(fig4[["library", "isolate", "mapped_pct", "mapped_fraction", "released",
                 "outcome", "internal_qc_decision", "y_jitter"]],
           "Fig4_panel_a_mapping_rate")

wrote:
  figures/Fig4_mapping_rate_distribution.pdf  (16 kB)
  figures/Fig4_mapping_rate_distribution.svg  (41 kB)
  figures/Fig4_mapping_rate_distribution.png  (48 kB)
panel data: figures/figure_data/Fig4_panel_a_mapping_rate.csv  (316 rows, 8 cols)
